In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'backend'))

import json
import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv('../backend/.env')
sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 120

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

DOCUMENT_ID    = 'REPLACE_WITH_YOUR_DOCUMENT_ID'
FAISS_BASE_DIR = '../backend/storage/faiss_index'
EMBEDDING_MODEL = 'text-embedding-3-small'

print('Setup complete ✓')

In [ ]:
index_dir   = os.path.join(FAISS_BASE_DIR, DOCUMENT_ID)
index       = faiss.read_index(os.path.join(index_dir, 'index.faiss'))
chunks_path = os.path.join(index_dir, 'chunks.json')

with open(chunks_path) as f:
    chunks = json.load(f)

print(f'FAISS index vectors : {index.ntotal}')
print(f'Chunks loaded       : {len(chunks)}')
print(f'Embedding dimension : {index.d}')
print(f'\nSample chunk:\n  page={chunks[0]["page"]} | text={chunks[0]["text"][:100]}...')

In [ ]:
def embed(text: str) -> np.ndarray:
    """Returns a normalised float32 embedding vector."""
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text.replace('\n', ' ')])
    vec  = np.array(resp.data[0].embedding, dtype='float32')
    return vec

def retrieve(query: str, k: int = 5) -> list[dict]:
    vec = embed(query).reshape(1, -1)
    distances, indices = index.search(vec, k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1: continue
        chunk = chunks[idx].copy()
        chunk['score'] = float(dist)
        results.append(chunk)
    return results

# Quick sanity check
sample_results = retrieve('What is quiz mode?', k=3)
for r in sample_results:
    print(f'Page {r["page"]} | score={r["score"]:.4f} | {r["text"][:120]}...')

In [ ]:
# Each entry: query + list of keywords that MUST appear in at least one retrieved chunk
eval_set = [
    {'query': 'What is quiz mode?',             'must_contain': ['quiz', 'question']},
    {'query': 'How does the RAG pipeline work?','must_contain': ['retriev', 'chunk', 'embed']},
    {'query': 'What is the technical stack?',   'must_contain': ['faiss', 'langchain', 'fastapi', 'openai']},
    {'query': 'What are the functional requirements?', 'must_contain': ['retriev', 'feedback', 'pdf']},
    {'query': 'Who are the target users?',      'must_contain': ['student', 'researcher', 'professional']},
    {'query': 'What is the project timeline?',  'must_contain': ['week', 'deploy', 'test']},
    {'query': 'How are embeddings generated?',  'must_contain': ['embed', 'openai', 'vector']},
    {'query': 'Explain chat mode features.',    'must_contain': ['chat', 'citation', 'source']},
]
print(f'Eval set size: {len(eval_set)} queries')

In [ ]:
def is_hit(chunks_retrieved: list[dict], must_contain: list[str]) -> bool:
    """Returns True if at least one keyword appears (case-insensitive) in any retrieved chunk."""
    combined = ' '.join(r['text'].lower() for r in chunks_retrieved)
    return any(kw.lower() in combined for kw in must_contain)

K_VALUES = [1, 3, 5, 10]
hit_rates = {k: [] for k in K_VALUES}

rows = []
for item in eval_set:
    row = {'query': item['query']}
    for k in K_VALUES:
        retrieved = retrieve(item['query'], k=k)
        hit       = is_hit(retrieved, item['must_contain'])
        hit_rates[k].append(hit)
        row[f'Hit@{k}'] = '✓' if hit else '✗'
    rows.append(row)

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

print('\n── Hit Rate Summary ──')
for k in K_VALUES:
    rate = sum(hit_rates[k]) / len(hit_rates[k]) * 100
    print(f'  Hit@{k:2d}: {rate:.1f}%')

In [ ]:
rates = [sum(hit_rates[k]) / len(hit_rates[k]) * 100 for k in K_VALUES]

plt.figure(figsize=(8, 4))
plt.plot(K_VALUES, rates, marker='o', color='#7c6af7', linewidth=2.5, markersize=8)
plt.fill_between(K_VALUES, rates, alpha=0.15, color='#7c6af7')
plt.axhline(y=80, color='green', linestyle='--', alpha=0.6, label='80% target')
plt.title('Hit Rate @ K (Retrieval Quality)')
plt.xlabel('K (number of chunks retrieved)')
plt.ylabel('Hit Rate (%)')
plt.xticks(K_VALUES)
plt.ylim(0, 105)
plt.legend()
plt.tight_layout()
plt.savefig('retrieval_hit_rate.png', bbox_inches='tight')
plt.show()

In [ ]:
def llm_judge(query: str, chunk_text: str) -> dict:
    """
    Asks GPT-4o to judge whether the chunk is relevant to the query.
    Returns {'relevant': bool, 'reason': str}.
    """
    prompt = (
        f'Query: {query}\n'
        f'Chunk: {chunk_text[:500]}\n\n'
        'Is this chunk relevant to the query? '
        'Respond in JSON: {"relevant": true/false, "reason": "one sentence"}'
    )
    resp = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
        max_tokens=120
    )
    raw = resp.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {'relevant': False, 'reason': 'Parse error'}


# Evaluate first 3 queries with LLM judge on top-3 chunks
print('Running LLM-as-Judge evaluation (3 queries × 3 chunks)...\n')
judge_results = []

for item in eval_set[:3]:
    results = retrieve(item['query'], k=3)
    for rank, r in enumerate(results):
        verdict = llm_judge(item['query'], r['text'])
        judge_results.append({
            'query'    : item['query'][:50] + '...',
            'rank'     : rank + 1,
            'page'     : r['page'],
            'l2_score' : round(r['score'], 4),
            'relevant' : '✓' if verdict['relevant'] else '✗',
            'reason'   : verdict['reason'],
        })

pd.DataFrame(judge_results)

In [ ]:
all_scores = []
for item in eval_set:
    for r in retrieve(item['query'], k=10):
        all_scores.append(r['score'])

plt.figure(figsize=(10, 4))
plt.hist(all_scores, bins=40, color='#a89cf7', edgecolor='white', alpha=0.85)
plt.axvline(x=np.median(all_scores), color='red',    linestyle='--',
            label=f'Median = {np.median(all_scores):.2f}')
plt.axvline(x=np.mean(all_scores),   color='orange', linestyle='--',
            label=f'Mean   = {np.mean(all_scores):.2f}')
plt.title('L2 Score Distribution (all queries, K=10)')
plt.xlabel('L2 Distance (lower = more similar)')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Min score  : {min(all_scores):.4f}')
print(f'Max score  : {max(all_scores):.4f}')
print(f'Std dev    : {np.std(all_scores):.4f}')

In [ ]:
# ── Fill in after reviewing Hit Rate @ K chart ──
RECOMMENDED_TOP_K = 5

hit_at_k = sum(hit_rates[RECOMMENDED_TOP_K]) / len(hit_rates[RECOMMENDED_TOP_K]) * 100

print('Recommended config to copy into backend/core/config.py:')
print(f'  TOP_K = {RECOMMENDED_TOP_K}')
print(f'\nHit Rate @ {RECOMMENDED_TOP_K}: {hit_at_k:.1f}%')
print(f'Assessment: {"✓ Acceptable" if hit_at_k >= 80 else "⚠ Below 80% — consider increasing TOP_K or re-chunking."}')